# Télécharger les données OpenStreetMap

Les fichiers IGN décrivent les limites administratives. **OpenStreetMap (OSM)** décrit notamment les rues, chemins et intersections qui serviront à calculer des déplacements à pied.

Nous utilisons **Geofabrik**, la source d'extraits OSM indiquée dans le sujet. Son découpage comprend encore deux anciennes régions : il faut **Auvergne et Rhône-Alpes** pour notre périmètre.

- [Catalogue France](https://download.geofabrik.de/europe/france.html)
- [Auvergne](https://download.geofabrik.de/europe/france/auvergne.html)
- [Rhône-Alpes](https://download.geofabrik.de/europe/france/rhone-alpes.html)
- Attribution : © OpenStreetMap contributors. [Licence ODbL 1.0](https://www.openstreetmap.org/copyright).

Le format **`.osm.pbf`** est un format binaire compact contenant les objets OSM et leurs attributs. Il n'est pas encore un graphe piéton : le filtrage et la construction du réseau viendront dans un prochain notebook.

Sélectionner le noyau **Python (IGN / OSM)** et exécuter dans l'ordre. Le premier téléchargement représente environ **690 Mo** ; les exécutions suivantes réutilisent les fichiers et contrôles locaux. Aucune nouvelle bibliothèque n'est nécessaire.

## 1. Préparer les chemins

Nous utilisons `pathlib` pour les chemins, `urllib` pour le téléchargement et `hashlib` pour calculer une empreinte du fichier. Une empreinte est une valeur calculée à partir du contenu : si le fichier change, elle change aussi.

In [1]:
from pathlib import Path
from urllib.request import urlopen
from datetime import date
import csv
import hashlib
import json

import geopandas as gpd
from IPython.display import HTML, display

current_directory = Path.cwd().resolve()
module_directory = None

for directory in [current_directory, *current_directory.parents]:
    if directory.name == "ign-osm" and (directory / "requirements.txt").is_file():
        module_directory = directory
        break
    if (directory / "ign-osm" / "requirements.txt").is_file():
        module_directory = directory / "ign-osm"
        break

if module_directory is None:
    raise FileNotFoundError("Open the notebook from the project directory.")

osm_directory = module_directory / "data" / "raw" / "osm"
osm_directory.mkdir(parents=True, exist_ok=True)
print(f"Data directory: {osm_directory}")

Data directory: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/data/raw/osm


## 2. Fixer les deux extraits

Les liens `latest` changent régulièrement. Nous choisissons les archives nommées **261016**, datées du 16 octobre 2026, pour fixer les fichiers utilisés. Ce nom désigne l'extrait Geofabrik ; l'horodatage précis des données OSM devra être lu dans le fichier lors de l'étape de lecture.

La liste contient deux dictionnaires Python, chacun décrivant un fichier. Les tailles attendues viennent du catalogue Geofabrik. Les fichiers MD5 associés serviront à vérifier le contenu.

In [2]:
base_url = "https://download.geofabrik.de/europe/france"
extracts = [
    {"name": "auvergne", "filename": "auvergne-261016.osm.pbf", "size_bytes": 156231713},
    {"name": "rhone-alpes", "filename": "rhone-alpes-261016.osm.pbf", "size_bytes": 531106658},
]
display(extracts)

[{'name': 'auvergne',
  'filename': 'auvergne-261016.osm.pbf',
  'size_bytes': 156231713},
 {'name': 'rhone-alpes',
  'filename': 'rhone-alpes-261016.osm.pbf',
  'size_bytes': 531106658}]

## 3. Définir une récupération vérifiable

La fonction suivante évite de répéter le même code pour les deux extraits :

1. Elle récupère le petit fichier MD5 et le conserve localement.
2. Elle télécharge le PBF par blocs de 1 Mo dans un fichier temporaire `.part`, s'il n'existe pas déjà.
3. Elle vérifie la taille et l'empreinte avant de renommer le téléchargement terminé.
4. Elle consigne la source, la date de récupération, les empreintes et la licence dans un fichier JSON associé au PBF.

`hashlib.file_digest()` lit le fichier par blocs sans le charger entièrement en mémoire. MD5 sert ici à détecter les erreurs de transfert, pas à garantir l'authenticité d'une source. Les données sont récupérées par HTTPS depuis Geofabrik.

Si un contrôle échoue, la cellule s'arrête. Ne pas modifier la taille attendue pour contourner une erreur : vérifier d'abord le fichier et sa source.

In [3]:
def download_extract(extract):
    source_url = f"{base_url}/{extract['filename']}"
    output_path = osm_directory / extract["filename"]
    checksum_path = osm_directory / f"{extract['filename']}.md5"
    metadata_path = osm_directory / f"{extract['filename']}.json"

    if not checksum_path.exists():
        with urlopen(f"{source_url}.md5", timeout=120) as response:
            checksum_path.write_bytes(response.read())
    expected_md5 = checksum_path.read_text().split()[0]
    if len(expected_md5) != 32 or any(character not in "0123456789abcdef" for character in expected_md5.lower()):
        raise ValueError("Invalid MD5 checksum file.")

    downloaded = not output_path.exists()
    candidate_path = output_path
    if downloaded:
        candidate_path = output_path.with_suffix(".pbf.part")
        print(f"Downloading {extract['filename']}...")
        with urlopen(source_url, timeout=120) as response:
            with candidate_path.open("wb") as destination:
                while chunk := response.read(1024 * 1024):
                    destination.write(chunk)
    else:
        print(f"Using existing file: {output_path.name}")

    if candidate_path.stat().st_size != extract["size_bytes"]:
        raise ValueError(f"Unexpected file size: {candidate_path}")
    with candidate_path.open("rb") as source_file:
        actual_md5 = hashlib.file_digest(source_file, "md5").hexdigest()
    if actual_md5 != expected_md5.lower():
        raise ValueError(f"MD5 mismatch: {candidate_path}")

    if downloaded:
        candidate_path.replace(output_path)

    with output_path.open("rb") as source_file:
        sha256 = hashlib.file_digest(source_file, "sha256").hexdigest()
    if downloaded or not metadata_path.exists():
        metadata = {
            "dataset": f"OpenStreetMap {extract['name']}",
            "publisher": "Geofabrik / OpenStreetMap contributors",
            "edition": "2026-10-16 (Geofabrik archive name)",
            "retrieved_on": date.today().isoformat() if downloaded else "unknown",
            "url": source_url,
            "local_path": str(output_path.relative_to(module_directory)),
            "size_bytes": output_path.stat().st_size,
            "md5": actual_md5,
            "sha256": sha256,
            "license_status": "ODbL-1.0; attribution OpenStreetMap contributors",
        }
        metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
    print(f"Verified: {output_path.name} ({output_path.stat().st_size:,} bytes)")
    return json.loads(metadata_path.read_text())

## 4. Récupérer les deux fichiers

La boucle appelle la même fonction pour chaque extrait. Attendre la fin de cette cellule avant de continuer ; elle peut durer plusieurs minutes. Le téléchargement ne charge pas les objets OSM en mémoire et ne construit pas encore les itinéraires.

In [4]:
records = []
for extract in extracts:
    records.append(download_extract(extract))
display(records)

Using existing file: auvergne-261016.osm.pbf


Verified: auvergne-261016.osm.pbf (156,231,713 bytes)
Using existing file: rhone-alpes-261016.osm.pbf


Verified: rhone-alpes-261016.osm.pbf (531,106,658 bytes)


[{'dataset': 'OpenStreetMap auvergne',
  'publisher': 'Geofabrik / OpenStreetMap contributors',
  'edition': '2026-10-16 (Geofabrik archive name)',
  'retrieved_on': '2026-10-19',
  'url': 'https://download.geofabrik.de/europe/france/auvergne-261016.osm.pbf',
  'local_path': 'data/raw/osm/auvergne-261016.osm.pbf',
  'size_bytes': 156231713,
  'md5': 'ee0e92abde0758604d75c9e58fbea8ec',
  'sha256': '16981cfc3e07c89235692531c5634d3d46bc3334ea73aea0a4769622f18923e0',
  'license_status': 'ODbL-1.0; attribution OpenStreetMap contributors'},
 {'dataset': 'OpenStreetMap rhone-alpes',
  'publisher': 'Geofabrik / OpenStreetMap contributors',
  'edition': '2026-10-16 (Geofabrik archive name)',
  'retrieved_on': '2026-10-19',
  'url': 'https://download.geofabrik.de/europe/france/rhone-alpes-261016.osm.pbf',
  'local_path': 'data/raw/osm/rhone-alpes-261016.osm.pbf',
  'size_bytes': 531106658,
  'md5': '69cb7ba35656390cda73554344ffe1c0',
  'sha256': '25d97f1eaa879aee199c899b8311cf740d4280d8539cd962a

## 5. Mettre à jour le registre des sources

Le CSV commun conserve les sources IGN et ajoute celles d'OSM. Nous remplaçons uniquement les lignes correspondant aux chemins des deux extraits : relancer cette cellule ne crée pas de doublons.

La date de récupération est celle du téléchargement, conservée dans le JSON local ; ce n'est pas la date de chaque exécution du notebook.

In [5]:
registry_path = module_directory / "sources.csv"
with registry_path.open(newline="", encoding="utf-8") as source_file:
    reader = csv.DictReader(source_file)
    fieldnames = reader.fieldnames
    existing_records = list(reader)

updated_paths = {record["local_path"] for record in records}
source_records = [record for record in existing_records if record["local_path"] not in updated_paths]
source_records.extend(records)

with registry_path.open("w", newline="", encoding="utf-8") as destination:
    writer = csv.DictWriter(destination, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(source_records)

print(f"Registered {len(records)} OSM sources in {registry_path}")

Registered 2 OSM sources in /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/sources.csv


## Bilan

Nous avons les deux extraits bruts, leurs contrôles d'intégrité et leur provenance. Les fichiers sont exclus de Git avec le dossier `data/`.

Ils peuvent se chevaucher aux frontières : il ne faudra pas concaténer leurs objets sans dédoublonner les identifiants OSM. Leurs contours d'extraction ne sont pas une garantie d'égalité exacte avec la limite IGN. La fusion et la gestion des frontières seront traitées avant des calculs régionaux.

**Prochaine étape :** lire un extrait et examiner quelques rues sur une petite zone avant de construire le réseau piéton. Aucun temps de marche n'est encore calculé.